In [ ]:
# Step 4 (Pembuatan & Pelatihan Model BiGRU - 5 Fitur)
# Terdiri atas: 
#   1. Slicing Fitur (Mengambil 5 Kolom: EAR, MAR, dan 3 Head Pose)
#   2. Stratified Splitting (secara aman pada label 1D)
#   3. Arsitektur Bidirectional GRU dengan Regularisasi L2 & Batch Normalization
#   4. Advanced Callbacks untuk mencegah overfitting

import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

# ==========================================
# 1. LOAD DATASET HASIL EKSTRAKSI STEP 3 & SLICING FITUR
# ==========================================
print("=== Memuat Fitur dan Label ===")
try:
    X_raw = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy')
    y = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy')
    
    # Ambil 5 fitur (Kolom 0: EAR, 1: MAR, 2-4: Head Pose Pitch, Yaw, Roll)
    X = X_raw[:, :, :5]
    
    print(f"Sukses memuat data!")
    print(f"Bentuk X Awal (Semua Fitur)  : {X_raw.shape}")
    print(f"Bentuk X Baru (5 Fitur)      : {X.shape}")
    print(f"Bentuk y (Sample,)           : {y.shape}")
except FileNotFoundError:
    print("❌ Error: File .npy tidak ditemukan di /kaggle/input/.")
    print("Pastikan file dataset sudah di-add ke notebook Anda.")
    sys.exit(1)

# ==========================================
# 2. PREPROCESSING & STRATIFIED SPLITTING
# ==========================================
# Bagi data menggunakan label mentah 1D (y) agar 'stratify' bekerja sempurna
# Pembagian data: 80% Train, 10% Validation, 10% Test
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=42, stratify=y_temp_raw
)

# Lakukan One-Hot Encoding SETELAH data terbagi (format probabilitas matriks)
y_train = to_categorical(y_train_raw, num_classes=3)
y_val = to_categorical(y_val_raw, num_classes=3)
y_test = to_categorical(y_test_raw, num_classes=3)

print("\n=== Distribusi Data ===")
print(f"Data Latih (Train)      : {X_train.shape} | Label: {y_train.shape}")
print(f"Data Validasi (Val)     : {X_val.shape} | Label: {y_val.shape}")
print(f"Data Uji (Test)         : {X_test.shape} | Label: {y_test.shape}")

# ==========================================
# 3. MEMBANGUN ARSITEKTUR BiGRU
# ==========================================
def build_bigru_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        
        # Layer 1: Bidirectional GRU Pertama
        Bidirectional(GRU(64, return_sequences=True, kernel_regularizer=l2(0.001))),
        BatchNormalization(),
        Dropout(0.4),
        
        # Layer 2: Bidirectional GRU Kedua
        Bidirectional(GRU(32, return_sequences=False)),
        BatchNormalization(),
        Dropout(0.4),
        
        # Layer 3: Dense Layer 1
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Layer 4: Dense Layer 2
        Dense(32, activation='relu'),
        
        # Layer Output: Softmax Klasifikasi Multi-kelas (3 Derajat Kantuk)
        Dense(3, activation='softmax')
    ])
    
    return model

input_shape = (X_train.shape[1], X_train.shape[2]) # (30, 5)
model = build_bigru_model(input_shape)

# Kompilasi Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 4. SETTING CALLBACKS
# ==========================================
# Nama file penyimpanan disesuaikan untuk model BiGRU
checkpoint_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_bigru.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_filename, monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# ==========================================
# 5. PROSES TRAINING MODEL
# ==========================================
print("\n=== Memulai Pelatihan Model BiGRU ===")
EPOCHS = 50
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

# ==========================================
# 6. EVALUASI DAN VISUALISASI KINERJA
# ==========================================
print("\n=== Evaluasi Akhir Model ===")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi Akhir pada Data Uji: {test_acc * 100:.2f}%")
print(f"Loss Akhir pada Data Uji   : {test_loss:.4f}")

# Plotting Loss dan Accuracy
plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='b', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Akurasi Model (BiGRU - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='b', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Loss Model (BiGRU - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 7. MATRIKS EVALUASI LANJUTAN
# ==========================================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']

print("\n=== Classification Report ===")
print(classification_report(y_test_raw, y_pred_classes, target_names=target_names))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_raw, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (BiGRU - 5 Fitur)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.show()

In [ ]:
import os
from IPython.display import display, FileLink

print("=== DAFTAR MODEL TERBAIK ===")

# Nama file model 5 fitur (EAR, MAR, HeadPose)
model_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_bigru.keras'

# 1. Buat link download untuk Model LSTM (5 Fitur)
if os.path.exists(model_filename):
    print("\n✅ Klik link di bawah untuk mendownload Model biGRU (5 Fitur: EAR, MAR, HeadPose):")
    display(FileLink(model_filename))
else:
    print(f"\n❌ File '{model_filename}' tidak ditemukan.")

In [ ]:
# Step 4 (Pembuatan & Pelatihan Model 1D-CNN - 5 Fitur)
# Terdiri atas: 
#   1. Slicing Fitur (Mengambil 5 Kolom: EAR, MAR, dan 3 Head Pose)
#   2. Stratified Splitting (secara aman pada label 1D)
#   3. Arsitektur 1D-CNN dengan Conv1D, MaxPooling, dan Flatten
#   4. Advanced Callbacks untuk mencegah overfitting

import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

# ==========================================
# 1. LOAD DATASET HASIL EKSTRAKSI STEP 3 & SLICING FITUR
# ==========================================
print("=== Memuat Fitur dan Label ===")
try:
    X_raw = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy')
    y = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy')
    
    # Ambil 5 fitur (Kolom 0: EAR, 1: MAR, 2-4: Head Pose Pitch, Yaw, Roll)
    X = X_raw[:, :, :5]
    
    print(f"Sukses memuat data!")
    print(f"Bentuk X Awal (Semua Fitur)  : {X_raw.shape}")
    print(f"Bentuk X Baru (5 Fitur)      : {X.shape}")
    print(f"Bentuk y (Sample,)           : {y.shape}")
except FileNotFoundError:
    print("❌ Error: File .npy tidak ditemukan di /kaggle/input/.")
    print("Pastikan file dataset sudah di-add ke notebook Anda.")
    sys.exit(1)

# ==========================================
# 2. PREPROCESSING & STRATIFIED SPLITTING
# ==========================================
# Bagi data menggunakan label mentah 1D (y) agar 'stratify' bekerja sempurna
# Pembagian data: 80% Train, 10% Validation, 10% Test
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=42, stratify=y_temp_raw
)

# Lakukan One-Hot Encoding SETELAH data terbagi (format probabilitas matriks)
y_train = to_categorical(y_train_raw, num_classes=3)
y_val = to_categorical(y_val_raw, num_classes=3)
y_test = to_categorical(y_test_raw, num_classes=3)

print("\n=== Distribusi Data ===")
print(f"Data Latih (Train)      : {X_train.shape} | Label: {y_train.shape}")
print(f"Data Validasi (Val)     : {X_val.shape} | Label: {y_val.shape}")
print(f"Data Uji (Test)         : {X_test.shape} | Label: {y_test.shape}")

# ==========================================
# 3. MEMBANGUN ARSITEKTUR 1D-CNN
# ==========================================
def build_1dcnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        
        # Layer 1: Konvolusi 1D Pertama
        # kernel_size=3 berarti melihat pola dari 3 frame berurutan sekaligus
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling1D(pool_size=2), # Mengurangi dimensi temporal (downsampling)
        Dropout(0.4),
        
        # Layer 2: Konvolusi 1D Kedua
        Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.4),
        
        # Meratakan output konvolusi (2D) menjadi vektor 1D untuk masuk ke Dense layer
        Flatten(),
        
        # Layer 3: Dense Layer 1
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Layer 4: Dense Layer 2
        Dense(32, activation='relu'),
        
        # Layer Output: Softmax Klasifikasi Multi-kelas (3 Derajat Kantuk)
        Dense(3, activation='softmax')
    ])
    
    return model

input_shape = (X_train.shape[1], X_train.shape[2]) # (30, 5)
model = build_1dcnn_model(input_shape)

# Kompilasi Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 4. SETTING CALLBACKS
# ==========================================
# Nama file penyimpanan disesuaikan untuk model 1D-CNN
checkpoint_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_1dcnn.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_filename, monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# ==========================================
# 5. PROSES TRAINING MODEL
# ==========================================
print("\n=== Memulai Pelatihan Model 1D-CNN ===")
EPOCHS = 50
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

# ==========================================
# 6. EVALUASI DAN VISUALISASI KINERJA
# ==========================================
print("\n=== Evaluasi Akhir Model ===")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi Akhir pada Data Uji: {test_acc * 100:.2f}%")
print(f"Loss Akhir pada Data Uji   : {test_loss:.4f}")

# Plotting Loss dan Accuracy
plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='b', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Akurasi Model (1D-CNN - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='b', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Loss Model (1D-CNN - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 7. MATRIKS EVALUASI LANJUTAN
# ==========================================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']

print("\n=== Classification Report ===")
print(classification_report(y_test_raw, y_pred_classes, target_names=target_names))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_raw, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (1D-CNN - 5 Fitur)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.show()

In [ ]:
import os
from IPython.display import display, FileLink

print("=== DAFTAR MODEL TERBAIK ===")

# Nama file model 5 fitur (EAR, MAR, HeadPose)
model_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_1dcnn.keras'

# 1. Buat link download untuk Model LSTM (5 Fitur)
if os.path.exists(model_filename):
    print("\n✅ Klik link di bawah untuk mendownload Model 1D-CNN (5 Fitur: EAR, MAR, HeadPose):")
    display(FileLink(model_filename))
else:
    print(f"\n❌ File '{model_filename}' tidak ditemukan.")

In [ ]:
# Step 4 (Pembuatan & Pelatihan Model Transformer - 5 Fitur)
# Terdiri atas: 
#   1. Slicing Fitur (Mengambil 5 Kolom: EAR, MAR, dan 3 Head Pose)
#   2. Stratified Splitting (secara aman pada label 1D)
#   3. Arsitektur Time-Series Transformer (Multi-Head Self-Attention)
#   4. Advanced Callbacks untuk mencegah overfitting

import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# ==========================================
# 1. LOAD DATASET HASIL EKSTRAKSI STEP 3 & SLICING FITUR
# ==========================================
print("=== Memuat Fitur dan Label ===")
try:
    X_raw = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy')
    y = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy')
    
    # Ambil 5 fitur (Kolom 0: EAR, 1: MAR, 2-4: Head Pose Pitch, Yaw, Roll)
    X = X_raw[:, :, :5]
    
    print(f"Sukses memuat data!")
    print(f"Bentuk X Awal (Semua Fitur)  : {X_raw.shape}")
    print(f"Bentuk X Baru (5 Fitur)      : {X.shape}")
    print(f"Bentuk y (Sample,)           : {y.shape}")
except FileNotFoundError:
    print("❌ Error: File .npy tidak ditemukan di /kaggle/input/.")
    print("Pastikan file dataset sudah di-add ke notebook Anda.")
    sys.exit(1)

# ==========================================
# 2. PREPROCESSING & STRATIFIED SPLITTING
# ==========================================
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=42, stratify=y_temp_raw
)

y_train = to_categorical(y_train_raw, num_classes=3)
y_val = to_categorical(y_val_raw, num_classes=3)
y_test = to_categorical(y_test_raw, num_classes=3)

print("\n=== Distribusi Data ===")
print(f"Data Latih (Train)      : {X_train.shape} | Label: {y_train.shape}")
print(f"Data Validasi (Val)     : {X_val.shape} | Label: {y_val.shape}")
print(f"Data Uji (Test)         : {X_test.shape} | Label: {y_test.shape}")

# ==========================================
# 3. MEMBANGUN ARSITEKTUR TRANSFORMER
# ==========================================
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.0):
    """
    Blok inti Transformer: Multi-Head Self-Attention dan Feed Forward Network
    dengan skip connections (residual) dan Layer Normalization.
    """
    # Bagian 1: Multi-Head Self-Attention
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    res = layers.Add()([x, inputs]) # Residual connection

    # Bagian 2: Feed Forward Network (Dense)
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Dense(ff_dim, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(inputs.shape[-1])(x)
    return layers.Add()([x, res]) # Residual connection

def build_transformer_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    # Linear projection untuk menaikkan dimensi awal (dari 5 fitur menjadi 64 dimensi)
    x = layers.Dense(64)(inputs)
    
    # Stacking beberapa blok Transformer Encoder
    x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.3)
    x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.3)
    
    # Global Average Pooling untuk meratakan (flatten) sekuens waktu menjadi 1D array
    x = layers.GlobalAveragePooling1D()(x)
    
    # Lapisan Dense Classifier Tradisional
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(32, activation='relu')(x)
    
    outputs = layers.Dense(3, activation='softmax')(x)
    
    return Model(inputs=inputs, outputs=outputs)

input_shape = (X_train.shape[1], X_train.shape[2]) # (30, 5)
model = build_transformer_model(input_shape)

# Kompilasi Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 4. SETTING CALLBACKS
# ==========================================
# Nama file penyimpanan disesuaikan untuk model Transformer
checkpoint_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_transformer.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_filename, monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

# ==========================================
# 5. PROSES TRAINING MODEL
# ==========================================
print("\n=== Memulai Pelatihan Model Transformer ===")
EPOCHS = 50
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

# ==========================================
# 6. EVALUASI DAN VISUALISASI KINERJA
# ==========================================
print("\n=== Evaluasi Akhir Model ===")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi Akhir pada Data Uji: {test_acc * 100:.2f}%")
print(f"Loss Akhir pada Data Uji   : {test_loss:.4f}")

# Plotting Loss dan Accuracy
plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='b', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Akurasi Model (Transformer - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='b', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Loss Model (Transformer - 5 Fitur)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 7. MATRIKS EVALUASI LANJUTAN
# ==========================================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']

print("\n=== Classification Report ===")
print(classification_report(y_test_raw, y_pred_classes, target_names=target_names))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_raw, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (Transformer - 5 Fitur)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.show()

In [ ]:
import os
from IPython.display import display, FileLink

print("=== DAFTAR MODEL TERBAIK ===")

# Nama file model 5 fitur (EAR, MAR, HeadPose)
model_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_transformer.keras'

# 1. Buat link download untuk Model LSTM (5 Fitur)
if os.path.exists(model_filename):
    print("\n✅ Klik link di bawah untuk mendownload Model Transformer (5 Fitur: EAR, MAR, HeadPose):")
    display(FileLink(model_filename))
else:
    print(f"\n❌ File '{model_filename}' tidak ditemukan.")

In [ ]:
# Step 4 (Pembuatan & Pelatihan Model XGBoost - 5 Fitur)
# Terdiri atas: 
#   1. Slicing Fitur (Mengambil 5 Kolom: EAR, MAR, dan 3 Head Pose)
#   2. Stratified Splitting (menggunakan label 1D)
#   3. FLATTENING: Meratakan 3D array (30, 5) menjadi 2D (150 fitur)
#   4. Training menggunakan XGBoost dengan Early Stopping
#   5. Ekstraksi Feature Importance

import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

# ==========================================
# 1. LOAD DATASET HASIL EKSTRAKSI STEP 3 & SLICING FITUR
# ==========================================
print("=== Memuat Fitur dan Label ===")
try:
    X_raw = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy')
    y = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy')
    
    # Ambil 5 fitur (Kolom 0: EAR, 1: MAR, 2-4: Head Pose Pitch, Yaw, Roll)
    X = X_raw[:, :, :5]
    
    print(f"Sukses memuat data!")
    print(f"Bentuk X Awal (Semua Fitur)  : {X_raw.shape}")
    print(f"Bentuk X Baru (5 Fitur)      : {X.shape}")
    print(f"Bentuk y (Sample,)           : {y.shape}")
except FileNotFoundError:
    print("❌ Error: File .npy tidak ditemukan di /kaggle/input/.")
    print("Pastikan file dataset sudah di-add ke notebook Anda.")
    sys.exit(1)

# ==========================================
# 2. PREPROCESSING & STRATIFIED SPLITTING
# ==========================================
# Bagi data menggunakan label mentah 1D (y) agar 'stratify' bekerja sempurna
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# ==========================================
# 3. FLATTENING UNTUK XGBOOST (3D -> 2D)
# ==========================================
# XGBoost membutuhkan input tabular 2D (Samples, Features)
# Kita ratakan dimensi Timesteps (30) x Features (5) menjadi 150 kolom
print("\n=== Meratakan Dimensi (Flattening) untuk XGBoost ===")
num_samples_train = X_train.shape[0]
num_samples_val = X_val.shape[0]
num_samples_test = X_test.shape[0]
num_flat_features = X_train.shape[1] * X_train.shape[2]

X_train_flat = X_train.reshape(num_samples_train, num_flat_features)
X_val_flat = X_val.reshape(num_samples_val, num_flat_features)
X_test_flat = X_test.reshape(num_samples_test, num_flat_features)

print(f"Bentuk Train setelah Flatten : {X_train_flat.shape}")
print(f"Bentuk Val setelah Flatten   : {X_val_flat.shape}")
print(f"Bentuk Test setelah Flatten  : {X_test_flat.shape}")

# ==========================================
# 4. INISIALISASI & TRAINING XGBOOST
# ==========================================
print("\n=== Memulai Pelatihan Model XGBoost ===")
# Inisialisasi Model XGBoost
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=3, 
    n_estimators=300,        
    learning_rate=0.05,      
    max_depth=6,             
    subsample=0.8,           
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=15  # <--- PINDAHKAN KE SINI
)

eval_set = [(X_train_flat, y_train), (X_val_flat, y_val)]

# Proses Training (tanpa early_stopping_rounds di dalam fit)
xgb_model.fit(
    X_train_flat, y_train, 
    eval_set=eval_set, 
    verbose=10 
)

# Simpan Model XGBoost 
checkpoint_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_xgboost.json'
xgb_model.save_model(checkpoint_filename)
print(f"\nModel tersimpan sebagai: {checkpoint_filename}")

# ==========================================
# 5. VISUALISASI KURVA LOSS (M-LogLoss)
# ==========================================
results = xgb_model.evals_result()
epochs = len(results['validation_0']['mlogloss'])
x_axis = range(0, epochs)

plt.figure(figsize=(8, 5))
plt.plot(x_axis, results['validation_0']['mlogloss'], label='Train Loss', color='b', linewidth=2)
plt.plot(x_axis, results['validation_1']['mlogloss'], label='Val Loss', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Log-Loss Model (XGBoost - 5 Fitur)')
plt.xlabel('Iterasi / Estimator')
plt.ylabel('Multi-class Log Loss')
plt.legend()
plt.grid(True)
plt.show()

# ==========================================
# 6. EVALUASI DAN VISUALISASI KINERJA
# ==========================================
print("\n=== Evaluasi Akhir Model ===")
# Prediksi data uji
y_pred_classes = xgb_model.predict(X_test_flat)

test_acc = accuracy_score(y_test, y_pred_classes)
print(f"Akurasi Akhir pada Data Uji: {test_acc * 100:.2f}%\n")

target_names = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']
print("=== Classification Report ===")
print(classification_report(y_test, y_pred_classes, target_names=target_names))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (XGBoost - 5 Fitur)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.show()

# ==========================================
# 7. FEATURE IMPORTANCE (Analisis Interpretasi)
# ==========================================
print("\n=== Plot Feature Importance ===")
# Menampilkan 20 fitur paling berpengaruh (dari 150 fitur hasil flattening)
plt.figure(figsize=(10, 8))
xgb.plot_importance(xgb_model, max_num_features=20, height=0.5, color='#10B981')
plt.title("Top 20 Fitur Paling Berpengaruh (Feature Importance)")
plt.show()

In [ ]:
import os
from IPython.display import display, FileLink

print("=== DAFTAR MODEL TERBAIK ===")

# Nama file model XGBoost 5 fitur (EAR, MAR, HeadPose)
model_filename = 'EAR_MAR_HeadPose_5000_frames_30_seq_best_drowsiness_xgboost.json'

# 1. Buat link download untuk Model XGBoost (5 Fitur)
if os.path.exists(model_filename):
    print("\n✅ Klik link di bawah untuk mendownload Model XGBoost (5 Fitur: EAR, MAR, HeadPose):")
    display(FileLink(model_filename))
else:
    print(f"\n❌ File '{model_filename}' tidak ditemukan.")

In [ ]:
# Step 4 (Pembuatan & Pelatihan Model BiLSTM - EAR Only)
# Terdiri atas: 
#   1. Slicing Fitur (Mengambil 1 Kolom: EAR)
#   2. Stratified Splitting (secara aman pada label 1D)
#   3. Arsitektur Bidirectional LSTM dengan Regularisasi L2 & Batch Normalization
#   4. Advanced Callbacks untuk mencegah overfitting

import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

# ==========================================
# 1. LOAD DATASET HASIL EKSTRAKSI STEP 3 & SLICING FITUR
# ==========================================
print("=== Memuat Fitur dan Label ===")
try:
    X_raw = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy')
    y = np.load('/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy')
    
    # Ambil HANYA fitur EAR (Kolom 0: EAR)
    X = X_raw[:, :, :1]
    
    print(f"Sukses memuat data!")
    print(f"Bentuk X Awal (Semua Fitur)  : {X_raw.shape}")
    print(f"Bentuk X Baru (EAR Only)     : {X.shape}")
    print(f"Bentuk y (Sample,)           : {y.shape}")
except FileNotFoundError:
    print("❌ Error: File .npy tidak ditemukan di /kaggle/input/.")
    print("Pastikan file dataset sudah di-add ke notebook Anda.")
    sys.exit(1)

# ==========================================
# 2. PREPROCESSING & STRATIFIED SPLITTING
# ==========================================
# Bagi data menggunakan label mentah 1D (y) agar 'stratify' bekerja sempurna
# Pembagian data: 80% Train, 10% Validation, 10% Test
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=42, stratify=y_temp_raw
)

# Lakukan One-Hot Encoding SETELAH data terbagi (format probabilitas matriks)
y_train = to_categorical(y_train_raw, num_classes=3)
y_val = to_categorical(y_val_raw, num_classes=3)
y_test = to_categorical(y_test_raw, num_classes=3)

print("\n=== Distribusi Data ===")
print(f"Data Latih (Train)      : {X_train.shape} | Label: {y_train.shape}")
print(f"Data Validasi (Val)     : {X_val.shape} | Label: {y_val.shape}")
print(f"Data Uji (Test)         : {X_test.shape} | Label: {y_test.shape}")

# ==========================================
# 3. MEMBANGUN ARSITEKTUR BiLSTM
# ==========================================
def build_bilstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        
        # Layer 1: Bidirectional LSTM Pertama
        Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001))),
        BatchNormalization(),
        Dropout(0.4),
        
        # Layer 2: Bidirectional LSTM Kedua
        Bidirectional(LSTM(32, return_sequences=False)),
        BatchNormalization(),
        Dropout(0.4),
        
        # Layer 3: Dense Layer 1
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Layer 4: Dense Layer 2
        Dense(32, activation='relu'),
        
        # Layer Output: Softmax Klasifikasi Multi-kelas (3 Derajat Kantuk)
        Dense(3, activation='softmax')
    ])
    
    return model

input_shape = (X_train.shape[1], X_train.shape[2]) # (30, 1)
model = build_bilstm_model(input_shape)

# Kompilasi Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 4. SETTING CALLBACKS
# ==========================================
# Nama file penyimpanan disesuaikan untuk model BiLSTM (EAR Only)
checkpoint_filename = 'EAR_only_5000_frames_30_seq_best_drowsiness_bilstm.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_filename, monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# ==========================================
# 5. PROSES TRAINING MODEL
# ==========================================
print("\n=== Memulai Pelatihan Model BiLSTM ===")
EPOCHS = 50
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

# ==========================================
# 6. EVALUASI DAN VISUALISASI KINERJA
# ==========================================
print("\n=== Evaluasi Akhir Model ===")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi Akhir pada Data Uji: {test_acc * 100:.2f}%")
print(f"Loss Akhir pada Data Uji   : {test_loss:.4f}")

# Plotting Loss dan Accuracy
plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='b', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Akurasi Model (BiLSTM - EAR Only)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='b', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='r', linestyle='--', linewidth=2)
plt.title('Kurva Loss Model (BiLSTM - EAR Only)', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 7. MATRIKS EVALUASI LANJUTAN
# ==========================================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']

print("\n=== Classification Report ===")
print(classification_report(y_test_raw, y_pred_classes, target_names=target_names))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_raw, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (BiLSTM - EAR Only)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.show()

In [ ]:
import os
from IPython.display import display, FileLink

print("=== DAFTAR MODEL TERBAIK ===")

# Nama file model 1 fitur (EAR)
model_filename = 'EAR_only_5000_frames_30_seq_best_drowsiness_bilstm.keras'

# 1. Buat link download untuk Model biLSTM (1 Fitur)
if os.path.exists(model_filename):
    print("\n✅ Klik link di bawah untuk mendownload Model biLSTM (1 Fitur: EAR):")
    display(FileLink(model_filename))
else:
    print(f"\n❌ File '{model_filename}' tidak ditemukan.")